# `shinka` Tutorial — Subscription-Based (Headless CLI) 🧬

This is a replica of [`shinka_tutorial.ipynb`](./shinka_tutorial.ipynb), but instead of
calling model **APIs** (which meter your tokens per call), it routes every LLM mutation
through the local **[Headless CLI](https://github.com/RobertTLange/headless-cli)** using
`headless/<agent>` model strings.

That means the mutation calls run against **agent CLIs you've already logged into with a
subscription** (Claude Code / Codex), so you spend plan quota rather than API credits.

**What changes vs. the original tutorial:**
- No `ANTHROPIC_API_KEY` / `GEMINI_API_KEY` needed for mutations — those run through Headless.
- Model strings become `headless/claude` and `headless/codex@gpt-5.5?effort=high`.
- Shinka shells out to `npx -y @roberttlange/headless`; for the `claude` agent it even
  strips `ANTHROPIC_API_KEY` from the subprocess so the CLI uses your logged-in session.
- **One exception — embeddings:** to keep the API tutorial's *novelty rejection*, we embed
  candidate programs, and there is no headless embedding route. So we use `OPENAI_API_KEY`
  for **embeddings only** (cheap); the novelty *judge* still runs through `headless/claude`.
  See section 3. Set `embedding_model=None` there to drop this and run fully key-free.

## 0. Prerequisites (do this once, outside the notebook)

The Headless CLI drives whichever agent CLI you point it at. Install + log in to at least one:

**Claude (`headless/claude`):**
```bash
npm install -g @anthropic-ai/claude-code   # or your preferred install
claude login                               # authenticate with your Claude subscription
```

**Codex (`headless/codex@...`):**
```bash
npm install -g @openai/codex
codex login                                # authenticate with your ChatGPT subscription
```

You also need **Node/npx** available, since Shinka calls `npx -y @roberttlange/headless` by default.
Override the command with the `SHINKA_HEADLESS_COMMAND` env var if you installed it differently,
and the per-call timeout with `SHINKA_HEADLESS_TIMEOUT` (seconds).

## 1. Repo / environment setup

Same as the original tutorial: make the repo importable. (Colab-aware.)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# detect colab
try:
    import google.colab  # type: ignore

    in_colab = True
except Exception:
    in_colab = False

repo_name = "ShinkaEvolve"
https_url = "https://github.com/SakanaAI/ShinkaEvolve.git"

cwd = Path.cwd()
repo_root = cwd

if not (repo_root / repo_name).exists():
    for parent in cwd.resolve().parents:
        if (parent / repo_name).exists():
            repo_root = parent
            break

if in_colab:
    root_candidate = Path("/content") / repo_name
    if not root_candidate.exists():
        print("cloning repository...")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", https_url, str(root_candidate)]
        )
    repo_root = root_candidate
    os.chdir(repo_root)
    print("installing package and deps...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

print("repo_root:", repo_root)

In [ ]:
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "shinka").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "shinka").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

# Mutations AND the novelty judge run through your logged-in agent CLIs
# (Headless), which need no *_API_KEY. The ONE exception is embedding extraction
# for novelty rejection (section 3): there is no headless embedding route, so we
# load OPENAI_API_KEY from the repo .env for that alone.
env_path = repo_root / ".env"
if env_path.exists():
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path)
        print("loaded .env")
    except Exception as e:
        print("could not load .env:", e)
else:
    print(".env not found; set OPENAI_API_KEY manually if you keep embeddings on")

# Sanity check: novelty rejection needs this. If it prints False, either add
# OPENAI_API_KEY to .env, or set evo_config.embedding_model=None below to
# disable novelty rejection and run fully key-free.
print("OPENAI_API_KEY available:", bool(os.getenv("OPENAI_API_KEY")))

## Shinka overview

Same core components as the API tutorial — only the mutation/judge backend changes
(Headless CLI instead of model APIs).

**Population & archive**
- `ProgramDatabase` stores *all* candidates, scores, and metadata.
- An **archive** keeps strong and diverse solutions across **islands**.
- Islands evolve in parallel to avoid premature convergence.

**Mutations**
- `patch_types`: `diff` (targeted edits), `full` (rewrite), `cross` (combine two parents).
- LLMs are prompted with task context and inspiration from the archive — here those
  LLMs are your logged-in `headless/<agent>` CLIs.

**Novelty**
- Optional semantic checks with embeddings to avoid duplicating ideas:
  `code_embed_sim_threshold`, `embedding_model`, `novelty_llm_models` (see section 3).
- Embeddings have no headless route, so this is the one place OpenAI is used.

**Meta-learning**
- Periodic reviews (`meta_rec_interval`) extract patterns into a scratchpad.
- Arm selection across multiple LLMs via bandits (e.g., `UCB1`).

## 2. Verify the Headless CLI is reachable

Shinka runs `headless --check` before evolution starts; we do the same explicitly here so
any login/toolchain problem surfaces now instead of mid-run. This is the subscription-based
equivalent of the original tutorial's "pick LLMs based on available keys" cell.

In [ ]:
from shinka.llm.providers.headless import (
    check_headless_available,
    headless_command_prefix,
    parse_headless_model,
)

print("headless command:", " ".join(headless_command_prefix()))

# Raises ValueError with a helpful message if npx/the CLI/login is missing.
check_headless_available()
print("\u2705 headless CLI is available")

### Check your subscription window before running

Shinka can read the same usage data Claude Code's `/usage` shows and **pause the
evolution loop automatically** when your 5-hour window is nearly exhausted
(instead of burning patch attempts on calls that are guaranteed to fail). This is
controlled by `subscription_pause_threshold` in `EvolutionConfig` (default `0.95`;
set `None` to disable). If the *weekly* cap is reached, new proposals stop for the
rest of the run, since that reset can be days away.

The cell below shows where you stand right now:

In [ ]:
from shinka.llm.subscription_usage import get_claude_usage

usage = get_claude_usage(cache_ttl=0.0)
if usage is None:
    print("Usage unavailable (no Claude Code login on this machine?) — gating will be a no-op.")
else:
    import datetime as _dt

    def _fmt(ts):
        return _dt.datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M") if ts else "?"

    print(f"5-hour window: {usage.five_hour_pct:.0f}% used, resets {_fmt(usage.five_hour_resets_at)}")
    print(f"weekly window: {usage.seven_day_pct:.0f}% used, resets {_fmt(usage.seven_day_resets_at)}")

## 3. Custom shinka configuration (headless models)

Mirrors `examples/circle_packing/run_evo.py` but with a small budget and **headless** model
strings. Add or remove entries in `llm_models` depending on which CLIs you logged into above.

Notes specific to headless:
- The `?effort=` query param controls reasoning effort (`low|medium|high|xhigh`) — it comes
  from the model string, not from `llm_kwargs`.
- `temperature` is never sent for headless models (the CLI owns sampling), so no `temperature`
  deprecation errors here.

**Novelty rejection (why one OpenAI key).** To replicate the API tutorial's novelty filtering,
we enable it here. It is a **two-stage gate**:

1. **Embedding similarity (cheap, no LLM).** Each candidate is embedded and compared (cosine)
   against the programs on its island. If the top similarity is `<= code_embed_sim_threshold`
   (0.99) the candidate is accepted immediately.
2. **LLM judge (rare).** Only when similarity *exceeds* the threshold does an LLM decide whether
   the candidate is genuinely novel. If not, it is rejected and re-sampled from a different
   parent — up to `max_novelty_attempts` times.

Two things to know:
- Rejection needs **both** `embedding_model` *and* `novelty_llm_models`. Setting only
  `embedding_model` computes/stores vectors (for the WebUI) but performs **no** rejection.
- There is no headless embedding route, so `embedding_model` uses OpenAI
  (`text-embedding-3-small`, cheap). The stage-2 judge stays on `headless/claude`, so it spends
  subscription quota, not API credits.

To run fully key-free, set `embedding_model=None` (and drop `novelty_llm_models`) below.

In [ ]:
import datetime as dt
from time import perf_counter
from shinka.core import ShinkaEvolveRunner, EvolutionConfig
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig

# default circle packing message - can be customized to your liking!
search_task_sys_msg = (
    "You are an expert mathematician specializing in circle packing problems "
    "and computational geometry. The best known result for the sum of radii "
    "when packing 26 circles in a unit square is 2.635.\n\n"
    "Key directions to explore:\n"
    "1. The optimal arrangement likely involves variable-sized circles\n"
    "2. A pure hexagonal arrangement may not be optimal due to edge effects\n"
    "3. The densest known circle packings often use a hybrid approach\n"
    "4. The optimization routine is critically important - simple physics-"
    "based models with carefully tuned parameters\n"
    "5. Consider strategic placement of circles at square corners and edges\n"
    "6. Place larger circles near the center and smaller near the edges\n"
    "7. Math literature suggests special arrangements for specific n\n"
    "8. You can use scipy.optimize to refine radii given fixed centers and "
    "constraints\n\n"
    "Be creative and try to find a new solution."
)

In [ ]:
# Subscription-backed models via the Headless CLI.
# Uncomment whichever agents you logged into in step 0.
llm_models = [
    "headless/claude",
    # "headless/codex@gpt-5.5?effort=high",
]

# sanity-check the model strings parse before we start
for m in llm_models:
    parse_headless_model(m)
print("llm_models:", llm_models)

# unique experiment directory
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_tag = f"{timestamp}_headless"

# Headless calls spawn a CLI agent per mutation and are serialized, so keep
# the budget small while you play. Bump num_generations once it works.
MAX_EVALUATION_JOBS = 1
MAX_PROPOSAL_JOBS = 1
MAX_DB_WORKERS = 1

evo_config = EvolutionConfig(
    task_sys_msg=search_task_sys_msg,
    patch_types=["diff", "full", "cross"],
    patch_type_probs=[0.6, 0.3, 0.1],
    num_generations=4,  # small for a quick subscription-based run
    max_patch_resamples=3,
    max_patch_attempts=3,
    job_type="local",
    language="python",
    # headless (subscription) models instead of API models
    llm_models=llm_models,
    llm_kwargs=dict(
        # temperatures/max_tokens are ignored for headless models (the CLI owns
        # sampling); kept here only so the config shape matches the API tutorial.
        temperatures=[0.0],
        max_tokens=16384,
        reasoning_efforts=["high"],
    ),
    # Embeddings for novelty rejection. There is no headless embedding route, so
    # this is the ONLY component that spends a metered API (OpenAI). It powers
    # the cheap cosine-similarity gate + WebUI clustering -- nothing else.
    embedding_model="text-embedding-3-small",
    # Novelty judge: fires ONLY when a candidate is > code_embed_sim_threshold
    # cosine-similar to an existing island program (rare), and runs through
    # Headless so it stays subscription-based. Rejected candidates are re-sampled
    # from a different parent, up to max_novelty_attempts times.
    novelty_llm_models=["headless/claude"],
    code_embed_sim_threshold=0.99,
    max_novelty_attempts=3,
    # no meta scratchpad
    meta_rec_interval=None,
    meta_llm_models=None,
    meta_llm_kwargs={},
    init_program_path="initial.py",
    results_dir=f"results/circle_packing/{run_tag}",
    llm_dynamic_selection="fixed",
)

db_config = DatabaseConfig(
    db_path="evolution_db.sqlite",
    num_islands=2,
    archive_size=20,
    elite_selection_ratio=0.3,
    num_archive_inspirations=4,
    num_top_k_inspirations=2,
    migration_interval=10,
    migration_rate=0.1,
    island_elitism=True,
    enforce_island_separation=True,
    parent_selection_strategy="weighted",
    parent_selection_lambda=10.0,
)

job_config = LocalJobConfig(eval_program_path="evaluate.py")

print("results_dir:", evo_config.results_dir)

## 4. Run the minimal circle-packing experiment

Identical to the original tutorial — only the models under the hood changed. The first mutation
may be slow while the agent CLI spins up.

In [ ]:
circle_packing_path = repo_root / "examples" / "circle_packing"
if os.getcwd() != str(circle_packing_path):
    os.chdir(circle_packing_path)
    print("changed working dir to:", circle_packing_path)

runner = ShinkaEvolveRunner(
    evo_config=evo_config,
    job_config=job_config,
    db_config=db_config,
    max_evaluation_jobs=MAX_EVALUATION_JOBS,
    max_proposal_jobs=MAX_PROPOSAL_JOBS,
    max_db_workers=MAX_DB_WORKERS,
    verbose=True,
)

tic = perf_counter()
await runner.run_async()
toc = perf_counter()

print("completed in", round(toc - tic, 2), "s")

## 5. Inspect the results

Load and plot the evolution trajectory and lineage tree of the best solution — same as the
API-based tutorial.

In [ ]:
import matplotlib.pyplot as plt

from shinka.utils import load_programs_to_df
from shinka.plots import plot_lineage_tree, plot_evals_performance

results_root = Path(runner.results_dir)

task_name = "Circle Packing with shinka (subscription/headless)"
if os.path.exists(f"{results_root}/{results_root}/programs.sqlite"):
    db_root = results_root / results_root
else:
    db_root = results_root

df = load_programs_to_df(f"{db_root}/programs.sqlite")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(30, 10), gridspec_kw={"width_ratios": [1, 1.5]})

fig.suptitle(
    f"{task_name}",
    fontsize=30,
    weight="bold",
    y=1,
)

plot_evals_performance(df, f"{task_name}: Improvements", fig, axs[0])
plot_lineage_tree(df, f"{task_name}: Evolution Tree", fig, axs[1])

plt.tight_layout()

## 6. Comparing parent selection strategies

Let's ablate one of the critical components of `shinka` — the **parent selection
strategy** — and compare against the weighted run above. Everything else (models,
budget, islands) is held fixed; only `parent_selection_strategy` flips to
`"uniform"`. Same experiment as the API tutorial, just with headless mutations.

In [ ]:
import copy

# Ablation: identical config but uniform (unweighted) parent selection.
db_config_uniform = copy.deepcopy(db_config)
db_config_uniform.parent_selection_strategy = "uniform"

evo_config_uniform = copy.deepcopy(evo_config)
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_tag = f"{timestamp}_uniform_headless"
evo_config_uniform.results_dir = f"results/circle_packing/{run_tag}"

circle_packing_path = repo_root / "examples" / "circle_packing"
if os.getcwd() != str(circle_packing_path):
    os.chdir(circle_packing_path)
    print("changed working dir to:", circle_packing_path)

# NOTE: unlike the API tutorial (which accidentally re-ran the *weighted* config
# here), we pass the *uniform* configs so the ablation is actually an ablation.
runner = ShinkaEvolveRunner(
    evo_config=evo_config_uniform,
    job_config=job_config,
    db_config=db_config_uniform,
    max_evaluation_jobs=MAX_EVALUATION_JOBS,
    max_proposal_jobs=MAX_PROPOSAL_JOBS,
    max_db_workers=MAX_DB_WORKERS,
    verbose=True,
)

tic = perf_counter()
await runner.run_async()
toc = perf_counter()

print("completed in", round(toc - tic, 2), "s")

In [ ]:
results_root_uniform = Path(runner.results_dir)
if os.path.exists(f"{results_root_uniform}/{results_root_uniform}/programs.sqlite"):
    db_root = results_root_uniform / results_root_uniform
else:
    db_root = results_root_uniform

task_name = "Shinka w/o parent weighting (uniform)"
df = load_programs_to_df(f"{db_root}/programs.sqlite")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(30, 10), gridspec_kw={"width_ratios": [1, 1.5]})

fig.suptitle(f"{task_name}", fontsize=30, weight="bold", y=1)
plot_evals_performance(df, f"{task_name}: Improvements", fig, axs[0])
plot_lineage_tree(df, f"{task_name}: Evolution Tree", fig, axs[1])
plt.tight_layout()

## 7. Launcher and preconfigured `shinka` configs

`shinka` ships many preset task/algorithm configs you can mix and match with the
launcher. Everything works the same in the subscription setting — just pass a
`headless/<agent>` model string wherever the API tutorial used an API model.

Shorthand launcher (bash):
```bash
shinka_launch \
    task=circle_packing \
    database=island_large \
    evolution=small_budget \
    cluster=local \
    evo_config.num_generations=10 \
    evo_config.llm_models='["headless/claude"]' \
    variant_suffix="_headless"
```

Or reuse an existing variant:
```bash
shinka_launch variant=circle_packing_example \
    evo_config.llm_models='["headless/claude"]'
```

Or load the presets from Python and override the models to headless:
```py
from shinka.utils.utils_hydra import build_cfgs_from_python

job_cfg, db_cfg, evo_cfg, cfg = build_cfgs_from_python(
    "variant=circle_packing_example",
    **{"evo_config.llm_models": ["headless/claude"]},
)

evo_runner = ShinkaEvolveRunner(
    evo_config=evo_cfg, job_config=job_cfg, db_config=db_cfg,
    max_evaluation_jobs=cfg.max_evaluation_jobs,
    max_proposal_jobs=cfg.max_proposal_jobs,
    max_db_workers=cfg.max_db_workers, verbose=cfg.verbose,
)
await evo_runner.run_async()
```

## 8. Novelty generator example (LLM-as-a-judge)

This shows `shinka` going beyond fixed metrics: an **LLM-as-a-judge** scores each
candidate on how *diverse*, *meaningful*, and *inspirational* its outputs are,
producing a `final_novelty_score`. We load the preset configs and override **both**
the mutation model *and* the judge to `headless/claude`, so the whole loop stays
subscription-based.

The judge runs once per program evaluation (all samples are batched into a single
prompt), so it stays cheap even with the preset's 20 samples. And — exactly like
the API tutorial — the `small_budget` preset keeps
`embedding_model="text-embedding-3-small"`, so embeddings (and only embeddings)
use OpenAI.

In [ ]:
from shinka.utils.utils_hydra import build_cfgs_from_python

launcher_args = [
    "variant=novelty_generator_example",
    "database=island_small",
    "evolution=small_budget",
    "evo_config.num_generations=10",  # API tutorial value; headless is slower
]

# Swap BOTH the mutation model and the LLM judge to headless (subscription).
launcher_kwargs = {
    "evo_config.llm_models": ["headless/claude"],
    "evaluate_function.llm_judge_names": ["headless/claude"],
}

if os.getcwd() != str(repo_root):
    os.chdir(repo_root)
    print("changed working dir to:", repo_root)

job_cfg, db_cfg, evo_cfg, cfg = build_cfgs_from_python(*launcher_args, **launcher_kwargs)
print("llm_models:", evo_cfg.llm_models)
print("embedding_model:", evo_cfg.embedding_model)

In [ ]:
evo_runner = ShinkaEvolveRunner(
    evo_config=evo_cfg,
    job_config=job_cfg,
    db_config=db_cfg,
    max_evaluation_jobs=cfg.max_evaluation_jobs,
    max_proposal_jobs=cfg.max_proposal_jobs,
    max_db_workers=cfg.max_db_workers,
    verbose=cfg.verbose,
)
await evo_runner.run_async()

### Inspecting results and loading the final function

In [ ]:
results_root = Path(evo_runner.results_dir)

task_name = "Novelty generator (headless)"
if os.path.exists(f"{results_root}/{results_root}/programs.sqlite"):
    db_root = results_root / results_root
else:
    db_root = results_root

df = load_programs_to_df(f"{db_root}/programs.sqlite")
fig, axs = plt.subplots(1, 2, figsize=(30, 10), gridspec_kw={"width_ratios": [1, 1.5]})

fig.suptitle(f"{task_name}", fontsize=30, weight="bold", y=1)
plot_evals_performance(df, f"{task_name}: Improvements", fig, axs[0])
plot_lineage_tree(df, f"{task_name}: Evolution Tree", fig, axs[1])
plt.tight_layout()

In [ ]:
import importlib.util
from rich.console import Console

console = Console()

program_path = results_root / "best/main.py"
spec = importlib.util.spec_from_file_location("program", program_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load module at {program_path}")

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

test_inputs = [1, 2, 3]
novel_outputs = module.run_experiment(test_inputs)
for art in novel_outputs:
    console.print(art)

### Customizing the novelty generator

Same customizations as the API tutorial, adapted to headless:
- give the agents a more explicit **procedural ASCII-art** system prompt,
- use **only `full` mutations** to push for diversity,
- keep the mutation model and judge on `headless/claude`.

In [ ]:
# More explicit system prompt: procedurally generated ASCII art.
new_system_prompt = (
    "Make a python function that takes as input a random integer and produces "
    "ASCII art that is cool, novel, and visually engaging. The art should be "
    "generated procedurally, with the random input seed controlling structures, "
    "patterns, and variations. Depending on its input, each output should be "
    "diverse from all other outputs produced with different inputs. Please, call "
    "this function \"def generate_novelty(rng: int) -> str\"\n\n"
    "Different judges will evaluate how 1) diverse, 2) meaningful, and 3) "
    "inspirational the generated ASCII art pieces are for different random seeds. "
    "These three criteria will be used to assign your function a "
    "\"final_novelty_score\" for each judge. Only functions excelling across all "
    "three dimensions will achieve a high \"final_novelty_score\".\n\n"
    "Now bring out your creativity, focus on procedural ASCII art, and surprise us!"
)

launcher_args = [
    "variant=novelty_generator_example",
    "database=island_small",
    "evolution=small_budget",
    "evo_config.num_generations=10",
]

# NOTE: unlike the API tutorial (which defined new_system_prompt but never used
# it), we actually wire it in via evo_config.task_sys_msg.
launcher_kwargs = {
    "evo_config.task_sys_msg": new_system_prompt,
    "evo_config.llm_models": ["headless/claude"],
    "evaluate_function.llm_judge_names": ["headless/claude"],
    "evo_config.patch_types": ["full"],
    "evo_config.patch_type_probs": [1],
}

job_cfg, db_cfg, evo_cfg, cfg = build_cfgs_from_python(*launcher_args, **launcher_kwargs)

In [ ]:
evo_runner = ShinkaEvolveRunner(
    evo_config=evo_cfg,
    job_config=job_cfg,
    db_config=db_cfg,
    max_evaluation_jobs=cfg.max_evaluation_jobs,
    max_proposal_jobs=cfg.max_proposal_jobs,
    max_db_workers=cfg.max_db_workers,
    verbose=cfg.verbose,
)
await evo_runner.run_async()

### Inspecting results of the custom implementation

In [ ]:
results_root = Path(evo_runner.results_dir)

task_name = "Novelty generator - ASCII art (headless)"
if os.path.exists(f"{results_root}/{results_root}/programs.sqlite"):
    db_root = results_root / results_root
else:
    db_root = results_root

df = load_programs_to_df(f"{db_root}/programs.sqlite")
fig, axs = plt.subplots(1, 2, figsize=(30, 10), gridspec_kw={"width_ratios": [1, 1.5]})

fig.suptitle(f"{task_name}", fontsize=30, weight="bold", y=1)
plot_evals_performance(df, f"{task_name}: Improvements", fig, axs[0])
plot_lineage_tree(df, f"{task_name}: Evolution Tree", fig, axs[1])
plt.tight_layout()

In [ ]:
console = Console()

program_path = results_root / "best/main.py"
spec = importlib.util.spec_from_file_location("program", program_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load module at {program_path}")

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

test_inputs = [1, 2, 3]
novel_outputs = module.run_experiment(test_inputs)
for art in novel_outputs:
    console.print(art)

## 9. Visualizing your runs with the WebUI

The WebUI is model-agnostic — it just reads the results DBs on disk, so it's
identical to the API tutorial.

On the **remote** machine where the run is stored:
```bash
shinka_visualize --port 8888
```
On your **local** machine (if remote != local), tunnel it:
```bash
ssh -L 8888:localhost:8888 your_user@remote-host
```
Then open <http://localhost:8888/>. The cells below launch it for a local setup.

In [ ]:
import subprocess, time

# start the webui as a background process
webui_proc = subprocess.Popen(
    ["shinka_visualize", "--port", "8888", "--open"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)
print("webui started on http://127.0.0.1:8888")

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src="http://127.0.0.1:8888", width="100%", height=800))

In [ ]:
webui_proc.terminate()

## Where to go next

You've now replicated the full API tutorial on a subscription backend: custom
config + circle packing, a parent-selection ablation, the launcher/presets, the
novelty-generator (LLM-as-judge) example and its ASCII-art customization, and the
WebUI.

- **Ensemble two subscription agents:** add `headless/codex@gpt-5.5?effort=high`
  to any `llm_models` list alongside `headless/claude`.
- **Scale up:** bump `num_generations` (kept small here because headless CLI calls
  are serialized and slower than API calls).
- **Only metered-API dependency anywhere above** is embedding extraction for
  novelty rejection (OpenAI `text-embedding-3-small`). Set `embedding_model=None`
  (and drop `novelty_llm_models`) to run fully key-free.
- **Env knobs:** `SHINKA_HEADLESS_COMMAND` (override the CLI invocation) and
  `SHINKA_HEADLESS_TIMEOUT` (per-call timeout in seconds).